# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to load, explore, and process the FAIR² dataset
using the `mlcroissant` library. All dataset elements are referenced by their unique `@id`.

### Dataset Source
The dataset is defined by its [Croissant JSON-LD schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).


In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

We load the dataset via `mlcroissant` using the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant metadata
dataset = mlc.Dataset(croissant_url)
print("Loaded dataset: [@id]", dataset.metadata.id)
print("Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)


## 2. Data Overview

Let's inspect available record sets in the dataset and summarize their fields, referencing all by `@id`.

The main data table in FAIR² is typically represented with a single record set. We'll list all available ones.

In [ ]:
# Examine record sets (by @id)
record_sets = list(dataset.record_sets)
print(f"Number of record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    # List fields for this record set
    print("  Fields:")
    for f in rs.fields:
        print(f"    - Field @id: {f.id}, Name: {f.name}, DataType: {f.data_type}")
    print()

## 3. Data Extraction

Now, we extract data from the main record set. All variables below are referenced by their `@id` as displayed above.


In [ ]:
# Select the main record set for extraction (replace the value with the @id found above if more than one)
main_record_set_id = record_sets[0].id

# For demonstration, collect all record set ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Use generator from dataset.records(record_set=@id)
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id {rs_id} with {df.shape[0]} rows and {df.shape[1]} columns.")
        print(f"Columns: {list(df.columns)}\n")
    else:
        print(f"No records found for RecordSet @id {rs_id}")

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field for demonstration. We'll filter rows, normalize, and group by another field, referencing all by `@id` or column name as shown previously.

In [ ]:
# Select the main DataFrame loaded from our primary record set
df = dataframes[main_record_set_id]

# Identify a numeric field by viewing columns. For demonstration, try 'Age_at_Second_Primary_Diagnosis' (replace if different)
candidate_numeric_fields = [c for c in df.columns if 'age' in c.lower() or df[c].dtype in [int, float]]
print("Candidate numeric fields:", candidate_numeric_fields)
numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.columns[0]

print(f"\nUsing numeric field: {numeric_field}")

# Example: filter for Age > 50 if it's the field, or otherwise threshold 10 as template
threshold = 50 if 'age' in numeric_field.lower() else 10
filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold} (showing first 5):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records (showing first 5):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping (e.g., by 'Sex' or first categorical field)
candidate_group_fields = [c for c in df.columns if c not in [numeric_field] and df[c].dtype == object]
group_field = ''
if candidate_group_fields:
    group_field = candidate_group_fields[0]

if group_field:
    print(f"\nGrouping by {group_field}:")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and (optionally) its relationship to a categorical field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.show()

# Boxplot by group field, if available
if group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion

This notebook demonstrated how to use `mlcroissant` to load and explore the FAIR² dataset, referencing all elements by their `@id`, extracting records, inspecting the dataset structure, and carrying out basic EDA and visualizations.

You may now proceed to more detailed analyses, feature engineering, or modeling, referencing fields and record sets consistently by their unique Croissant `@id`s.